In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

# Generate Mock Data
n_logs = 10000
user_ids = [f"USR_{np.random.randint(1000, 1500)}" for _ in range(n_logs)]
product_ids = [f"PRD_{np.random.randint(100, 200)}" for _ in range(n_logs)]
categories = ['Electronics', 'Clothing', 'Home', 'Books', 'Beauty']
product_category_map = {f"PRD_{i}": np.random.choice(categories) for i in range(100, 200)}

event_types = ['view', 'add_to_cart', 'purchase']
event_weights = [0.70, 0.22, 0.08] # Imbalanced: Most interactions are views

start_date = datetime(2026, 1, 1)
timestamps = [start_date + timedelta(minutes=int(x)) for x in np.random.randint(0, 60*24*60, n_logs)]

df_raw = pd.DataFrame({
    'user_id': user_ids,
    'product_id': product_ids,
    'event_type': np.random.choice(event_types, p=event_weights, size=n_logs),
    'created_at': timestamps
})

df_raw['category'] = df_raw['product_id'].map(product_category_map)
df_raw = df_raw.sort_values('created_at').reset_index(drop=True)

print("Raw Clickstream Logs Sample:")
print(df_raw.head())

Raw Clickstream Logs Sample:
    user_id product_id   event_type          created_at     category
0  USR_1344    PRD_158  add_to_cart 2026-01-01 00:14:00  Electronics
1  USR_1271    PRD_172         view 2026-01-01 00:16:00     Clothing
2  USR_1499    PRD_185         view 2026-01-01 00:18:00       Beauty
3  USR_1275    PRD_123  add_to_cart 2026-01-01 00:29:00     Clothing
4  USR_1040    PRD_116         view 2026-01-01 00:36:00  Electronics


In [2]:
import pandas as pd
import numpy as np

# 1. Define Binary Target (y)
# Convert event types: 'add_to_cart' or 'purchase' -> 1 (Conversion), 'view' -> 0
df_raw['target'] = df_raw['event_type'].apply(lambda x: 1 if x in ['add_to_cart', 'purchase'] else 0)

# Sort strictly by time to maintain temporal integrity
df_raw = df_raw.sort_values('created_at').reset_index(drop=True)

# 2. Extract Valid Time & Behavioral Features (X)
# Feature A: User total interactions prior to current row (Cumulative sum)
df_raw['user_prev_interactions'] = df_raw.groupby('user_id').cumcount()

# Feature B: Category popularity up to this point
df_raw['cat_interaction_count'] = df_raw.groupby('category').cumcount()

# Feature C: Extract temporal signals from timestamp
df_raw['hour_of_day'] = df_raw['created_at'].dt.hour
df_raw['day_of_week'] = df_raw['created_at'].dt.dayofweek

# 3. Clean Feature Matrix
features = ['user_prev_interactions', 'cat_interaction_count', 'hour_of_day', 'day_of_week']
X = df_raw[features]
y = df_raw['target']

print(f"Dataset Target Distribution (Imbalance Check):")
print(y.value_counts(normalize=True))

Dataset Target Distribution (Imbalance Check):
target
0    0.7016
1    0.2984
Name: proportion, dtype: float64


In [3]:
%pip install lightgbm

import lightgbm as lgb
%pip install scikit-learn

from sklearn.metrics import roc_auc_score

# Temporal Split (Train on past 80%, Test on future 20% to avoid leakage)
split_idx = int(len(df_raw) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Train LightGBM Classifier with balanced class weights
%pip install -U scikit-learn lightgbm

model = lgb.LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    random_state=42,
    verbosity=-1
)

model.fit(X_train, y_train)

# Predict probabilities instead of raw binary predictions
y_probs = model.predict_proba(X_test)[:, 1]

print(f"ROC-AUC Score: {roc_auc_score(y_test, y_probs):.4f}")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
ROC-AUC Score: 0.5082


In [4]:
%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer  # type: ignore[reportMissingImports]
import numpy as np

# Load a lightweight, high-speed embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# E-commerce Catalog Descriptions
catalog = [
    {"id": "PRD_101", "text": "Lightweight linen summer shirt breathable casual fit"},
    {"id": "PRD_102", "text": "Heavy wool winter jacket insulated waterproof coat"},
    {"id": "PRD_103", "text": "Ergonomic wireless gaming mouse RGB lighting"},
    {"id": "PRD_104", "text": "Running sneakers athletic cushioning lightweight mesh"},
]

# Generate Vector Embeddings (Matrix Shape: N x 384)
descriptions = [item['text'] for item in catalog]
embeddings = embedder.encode(descriptions, convert_to_numpy=True)

print(f"Generated Vector Matrix Shape: {embeddings.shape}")

Note: you may need to restart the kernel to use updated packages.


c:\Users\alaas\anaconda3\envs\nlp_project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\alaas\anaconda3\envs\nlp_project\lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\alaas\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see thi

Generated Vector Matrix Shape: (4, 384)


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 1. Product Catalog Embeddings (Pre-computed)
catalog = [
    {"id": "PRD_101", "text": "Lightweight linen summer shirt breathable casual fit"},
    {"id": "PRD_102", "text": "Heavy wool winter jacket insulated waterproof coat"},
    {"id": "PRD_103", "text": "Ergonomic wireless gaming mouse RGB lighting"},
    {"id": "PRD_104", "text": "Running sneakers athletic cushioning lightweight mesh"},
]
doc_vectors = embedder.encode([item['text'] for item in catalog], normalize_embeddings=True)

# 2. Live User Query Embedding
user_query = "something lightweight to wear in hot weather"
query_vector = embedder.encode([user_query], normalize_embeddings=True)[0]

# 3. Fast Dot Product Search (Since vectors are normalized)
scores = np.dot(doc_vectors, query_vector)

# Sort candidates by score
ranked_results = sorted(zip(catalog, scores), key=lambda x: x[1], reverse=True)

for item, score in ranked_results:
    print(f"Score: {score:.4f} | Product: {item['text']}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3218.46it/s]


Score: 0.5702 | Product: Lightweight linen summer shirt breathable casual fit
Score: 0.5435 | Product: Heavy wool winter jacket insulated waterproof coat
Score: 0.4370 | Product: Running sneakers athletic cushioning lightweight mesh
Score: 0.1710 | Product: Ergonomic wireless gaming mouse RGB lighting
